# Flight Price Prediction: Random Forest Model

This notebook trains a **Random Forest Regressor** to predict flight ticket prices. It is structured into multiple steps: data loading, preprocessing, training, evaluation, and feature importance analysis.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
data = pd.read_csv("c:/Users/Youcode/Desktop/model_billets_avion/data/Clean_Dataset.csv")

In [9]:

x = data.drop(columns=['price' , 'Unnamed: 0'])
y = data["price"]

In [10]:
X_train , X_test , Y_train , Y_test = train_test_split(
    x , y , random_state=42 , test_size=0.2
)

In [12]:
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
cat_cols = x.select_dtypes(include=['object', 'category', 'string']).columns


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder
preprocessor = ColumnTransformer(
    transformers= [
        ('num' , StandardScaler() , num_cols ),
        ('cat' , OneHotEncoder(handle_unknown='ignore') , cat_cols)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline 
from sklearn.ensemble import RandomForestRegressor
pipeline = Pipeline(
    steps=[
        ('preprocessor' , preprocessor),
        ('model' , RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42))
    ]
)

pipeline.fit(X_train, Y_train)



In [17]:
new_flight = pd.DataFrame([{
    "airline": "SpiceJet",
    "flight": "SG-8709",
    "source_city": "Delhi",
    "departure_time": "Evening",
    "stops": "zero",
    "arrival_time": "Night",
    "destination_city": "Mumbai",
    "class": "Economy",
    "duration": 2.17,
    "days_left": 1
}])

prediction = pipeline.predict(new_flight)

print("Prix réel :", 5953)
print("Prix prédit :", prediction[0])

Prix réel : 5953
Prix prédit : 6555.7


In [16]:

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from numpy import sqrt
y_pred = pipeline.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
rmse = sqrt(mse)
mae = mean_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.4f}")

MSE  : 5643789.52
RMSE : 2375.67
MAE  : 857.99
R²   : 0.9891


In [29]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    pipeline,
    "../models/best_model.joblib",
    compress=3
)

print("Pipeline sauvegardé avec succès.")

Pipeline sauvegardé avec succès.


In [30]:
import joblib

loaded_pipeline = joblib.load("../models/best_model.joblib")

print("Pipeline chargé avec succès !")

sample = X_test.iloc[[0]]

real_price = Y_test.iloc[0]

prediction = loaded_pipeline.predict(sample)

print("Prix réel :", real_price)
print("Prix prédit :", prediction[0])

Pipeline chargé avec succès !
Prix réel : 7366
Prix prédit : 7245.61
